In [30]:
%reload_ext autoreload
%autoreload 2

import os
import sys

import numpy as np
import healpy as hp

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

wdir = "/n/home07/yitians/fermi/fermi-prob-prog/production"
sys.path.append(f"{wdir}/..")
from models.np_model import NPModel

In [28]:
from dataclasses import dataclass

@dataclass
class Args:
    data: str
    model: str
    n_exp: int
    i: int
    fit_type: str
    n_step: int
    seed: int
    n: int

args = Args(
    data="base23fix_deltapsf",
    model="base23fix_deltapsf",
    n_exp=2,
    i=0,
    fit_type="svi",
    n_step=100,
    seed=4242,
    n=10000,
)

In [7]:
6839 % 7

0

In [ ]:
wdir = "/n/home07/yitians/fermi/fermi-prob-prog/production"
run_name = "test"

save_dir = f"{wdir}/../outputs/fit/{run_name}"
os.makedirs(save_dir, exist_ok=True)

mask_roi = np.load(f"{wdir}/mask_roi.npy")
mask_norm = jnp.load(f"{wdir}/mask_norm.npy")

# Ensure mask_roi's length is divisible by args.n_exp
n_pix_remainder = int(np.sum(~mask_roi)) % args.n_exp
if n_pix_remainder != 0:
    unmasked_indices = np.where(mask_roi == 0)[0]
    mask_roi[unmasked_indices[-n_pix_remainder:]] = 1

data = np.load(f"../outputs/sims/{args.data}_n100.npy")[args.i]
if len(data) < hp.nside2npix(128):
    data_full = np.zeros(hp.nside2npix(128))
    data_full[~mask_norm] = data
    data_in = jnp.array(data_full, dtype=jnp.int32)
else:
    data_in = jnp.array(data, dtype=jnp.int32)


psf_tag = 'delta' if 'deltapsf' in args.model else 'king'
print(f'USING PSF {psf_tag}')
m = NPModel(data=data_in, psf_tag=psf_tag, n_exp=args.n_exp, custom_mask_roi=mask_roi)

USING PSF delta
Number of exposure regions: 2
Number of pixels in ROI: 6838
Using psf: delta
Max photon count is 118


In [35]:
m.fit_svi(
    n_steps=args.n_step, data=data_in, lr=1e-4,
    rng_key=jax.random.PRNGKey(args.seed)
)
samples = m.get_svi_samples(num_samples=args.n)

100%|██████████| 100/100 [00:56<00:00,  1.78it/s, init loss: 28502.5808, avg. loss [96-100]: 25742.2805]
